In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from datetime import datetime

In [6]:
df_fakes = pd.read_excel("fake_news_dataset.xlsx")
df_reals = pd.read_csv("data/kyrgyz_news_balanced_1000.csv")

In [7]:
df_fakes.head()

,headline,body_text,source,url,date,language,label
0,Көчөгө белгисиз зат чачылган тасма Кыргызстанд...,Социалдык тармактарда ири унаа түнкүсүн көчөгө...,Factcheck.kg,https://factcheck.kg/ky/raspylenie-neizvestnog...,10.07.2020,kyrgyz,1
1,Кумтөр” 30 жылдыгына карата белектерди ойнотко...,“Кумтөр Голд Компани” 30 жылдыгына карата беле...,Factcheck.kg,https://factcheck.kg/ky/https-factcheck-kg-roz...,09.03.2022,kyrgyz,1
2,Камчыбек Ташиевдин сүрөтүндө жанында чын эле б...,ЖМКларда Камчыбек Ташиевдин сүрөтү жарыяланды....,Factcheck.kg,https://factcheck.kg/ky/faktchek-na-foto-kamch...,21.02.2023,kyrgyz,0
3,Аймактык медиалар эмне үчүн өнүкпөй жатат,"Элеттиктер районунда, айыл өкмөтүндө болуп жат...",Factcheck.kg,https://factcheck.kg/ky/kyrgyz-ajmaktyk-medial...,15.11.2022,kyrgyz,0
4,Адахан Мадумаров: Бир жылда 220 миң жаран өлкө...,Облус арасындагы миграция тууралуу маалымат ач...,Factcheck.kg,https://factcheck.kg/ky/adahan-madumarov-esli-...,28.12.2020,kyrgyz,0


In [8]:
df_reals.head()

,headline,body_text,source,url,date,language,label,word_count
0,"Быйыл тоо-кен тармагынан түшкөн салыктар 18,7 ...",2025-жылдын алгачкы 11 айында алдыңкы тармакта...,24.kg,https://24.kg/kyrgyzcha/356170_byiyyil_too-ken...,NaN,kyrgyz,0,NaN
1,Шаардык кеңешке кайсы партиялар ат салышты? Те...,"14-июнда Бишкек, Ош жана Токмок шаарларына ат ...",factcheck.kg,https://factcheck.kg/ky/kakie-partii-povtorno-...,NaN,kyrgyz,0,NaN
2,Каныбек Иманалиев: Бүгүнкү күндө сот системасы...,"Конституциягя ылайык, президент 1) Жогорку сот...",factcheck.kg,https://factcheck.kg/ky/kanybek-imanaliev-na-s...,NaN,kyrgyz,0,NaN
3,АКШ Европаны террористтик коркунучтардын очогу...,"БИШКЕК, 7-май — Sputnik.АКШнын терроризмге кар...",sputnik.kg,https://sputnik.kg/20260507/aksh-evropa-terror...,12:47 07.05.2026,kyrgyz,0,NaN
4,Атамбаевдин үй-бүлөсү турак-жайсыз калдыбы? Те...,Өткөн аптада укук коргоо органдары тарабынан м...,factcheck.kg,https://factcheck.kg/ky/imushhestvo-atambaeva-...,NaN,kyrgyz,0,NaN


In [10]:
REF_DATE = datetime(2026, 5, 9)

def unify_date(date_str):
    """
    Standardizes Kyrgyz/Russian relative dates and standard date strings.
    Converts everything to YYYY-MM-DD format.
    """
    if pd.isna(date_str) or str(date_str).lower() == 'nan':
        return None
    
    date_str = str(date_str).strip()
    
    # 1. Handle relative years (e.g., "5 лет мурун", "3 года мурун", "2 жыл мурун")
    # Rule: If X years ago, pick July 1st of that year.
    relative_match = re.search(r'(\d+)\s+(лет|года|жыл)\s+мурун', date_str, re.IGNORECASE)
    if relative_match:
        years_ago = int(relative_match.group(1))
        target_year = REF_DATE.year - years_ago
        return f"{target_year}-07-01"

    # 2. Handle ISO 8601 or standard formats using pandas parser
    try:
        # This handles '2026-04-15T11:07:17+06:00' and 'DD.MM.YYYY'
        dt = pd.to_datetime(date_str, dayfirst=True, errors='coerce')
        if not pd.isna(dt):
            return dt.strftime('%Y-%m-%d')
    except:
        pass

    # 3. Fallback regex for DD.MM.YYYY if pandas fails
    match_dmy = re.search(r'(\d{2})\.(\d{2})\.(\d{4})', date_str)
    if match_dmy:
        d, m, y = match_dmy.groups()
        return f"{y}-{m}-{d}"

    return None

def clean_kyrgyz_text(text):
    """
    Cleans text by removing fact-checking boilerplate and normalizing whitespace.
    """
    if not isinstance(text, str):
        return ""
    
    # Phrases specific to Kyrgyz fact-checkers that introduce bias
    boilerplate = [
        r"“?ПолитКлиника”? медиасы бул маалыматты текшерип көрдү\.?",
        r"Factcheck\.kg бул маалыматты текшерип көрдү\.?",
        r"Жыйынтыгы: (Калп|Манипуляция|Чындык)\.?",
        r"Вердикт: (Калп|Чындык|Манипуляция)\.?",
        r"Бул боюнча.*билдирүү жарыялаган\.",
        r"Бул маалыматты текшерип көрдүк\.?",
    ]
    
    for pattern in boilerplate:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    
    # Remove URLs and extra spaces
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def create_final_dataset(fakes_path, reals_path):
    # Load
    df_fakes = pd.read_excel("data/fake_news_dataset.xlsx")
    df_reals = pd.read_csv("data/kyrgyz_news_balanced_1000.csv")
    
    # Merge
    df = pd.concat([df_fakes, df_reals], ignore_index=True)
    
    # 1. Date Unification
    print("Unifying dates...")
    df['date_unified'] = df['date'].apply(unify_date)
    
    # 2. Text Cleaning
    print("Cleaning text...")
    df['body_text'] = df['body_text'].apply(clean_kyrgyz_text)
    df['headline'] = df['headline'].apply(clean_kyrgyz_text)
    
    # 3. Deduplication
    df = df.drop_duplicates(subset=['headline', 'body_text']).reset_index(drop=True)
    
    # 4. Filter empty/short results
    df = df[df['body_text'].str.len() > 15].reset_index(drop=True)
    
    # 5. Class Balancing (1:1 Ratio)
    fakes = df[df['label'] == 1]
    reals = df[df['label'] == 0]
    
    n_samples = min(len(fakes), len(reals))
    print(f"Balancing dataset to {n_samples} samples per class.")
    
    df_final = pd.concat([
        fakes.sample(n=n_samples, random_state=42),
        reals.sample(n=n_samples, random_state=42)
    ]).sample(frac=1, random_state=42).reset_index(drop=True)
    
    # Update word count for the cleaned text
    df_final['word_count'] = df_final['body_text'].apply(lambda x: len(x.split()))
    
    return df_final

# Execute
if __name__ == "__main__":
    final_dataset = create_final_dataset(
        'fake_news_dataset.xlsx - Sheet1.csv', 
        'kyrgyz_news_balanced_1000.csv'
    )
    
    # Save result
    final_dataset.to_csv('kyrgyz_news_final_unified.csv', index=False)
    print("Success! Final dataset saved as 'kyrgyz_news_final_unified.csv'")

Unifying dates...


C:\Users\User\AppData\Local\Temp\ipykernel_22076\1107794463.py:24: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(date_str, dayfirst=True, errors='coerce')


Cleaning text...
Balancing dataset to 841 samples per class.
Success! Final dataset saved as 'kyrgyz_news_final_unified.csv'
